# Week 11: Bayes' Theorem // Entropy Probability Distribution

This notebook walks through the Week 11 assignment in three parts:

- Use Bayes’ Theorem to “reverse” conditional probabilities using the probability tree.
- Write a reusable entropy function for a discrete probability distribution.
- Use that function to compare the entropy of two given distributions.

In [57]:
import numpy as np

#

## 1) Bayes’ Theorem (Process tree)

Bayes’ Theorem helps when I know probabilities like P(defective | process), but the question asks for the reverse like P(process | defective).

The basic idea:
- First compute the overall probability of the condition (like “defective”).
- Then divide the piece I care about by that total.

### Bayes’ Theorem

Bayes’ Theorem is:

- P(A | B) = P(B | A) · P(A) / P(B)

In this problem:
- A is the process (A, B, or C)
- B is the evidence (Defective or Not Defective)

example:
- **P(A | Def) = P(Def | A) · P(A) / P(Def)**

The key step is computing **P(Def)** using the tree:
- **P(Def) = P(A)P(Def|A) + P(B)P(Def|B) + P(C)P(Def|C)**

In [61]:
# Probabilities from the given tree diagram
P_A, P_B, P_C = 0.5, 0.3, 0.2

P_def_given_A = 0.03
P_def_given_B = 0.02
P_def_given_C = 0.04

# Complements (not defective) come from 1 - defective
P_not_def_given_A = 1 - P_def_given_A
P_not_def_given_B = 1 - P_def_given_B
P_not_def_given_C = 1 - P_def_given_C

### Plan

To answer the Bayes questions, I’m going to compute:

- **P(defective)** using the law of total probability  
- **P(A | defective)** using Bayes  
- **P(not defective)** the same way  
- **P(C | not defective)** using Bayes

In [64]:
# Total probability of getting a defective card (adds up defect rates across processes)
P_def = (P_A * P_def_given_A) + (P_B * P_def_given_B) + (P_C * P_def_given_C)

# Bayes: probability it came from A given that it is defective
P_A_given_def = (P_A * P_def_given_A) / P_def

# Total probability of getting a non-defective card
P_not_def = (P_A * P_not_def_given_A) + (P_B * P_not_def_given_B) + (P_C * P_not_def_given_C)

# Bayes: probability it came from C given that it is not defective
P_C_given_not_def = (P_C * P_not_def_given_C) / P_not_def

P_def, P_A_given_def, P_not_def, P_C_given_not_def

(0.028999999999999998,
 0.5172413793103449,
 0.9709999999999999,
 0.1977342945417096)

In [66]:
# Print rounded values so the final answers are easy to copy into the assignment
print(f"P(Def) = {P_def:.4f}")
print(f"P(A | Def) = {P_A_given_def:.4f}")
print(f"P(Not Def) = {P_not_def:.4f}")
print(f"P(C | Not Def) = {P_C_given_not_def:.4f}")

P(Def) = 0.0290
P(A | Def) = 0.5172
P(Not Def) = 0.9710
P(C | Not Def) = 0.1977


### Interpretation (Bayes results)

The overall probability of getting a defective card is:

- P(Def) = 0.0290, which means about 2.9% of cards are defective across all processes.

Using Bayes’ Theorem:

- P(A | Def) = 0.5172
  
This means that if I randomly select a defective card, there is about a 51.7% chance it came from Process A. Even though Process A produces only half the cards, it accounts for the largest share of defects.

For non-defective cards:

- **P(Not Def) = 0.9710**
- **P(C | Not Def) = 0.1977**

So if a card is not defective, there is about a 19.8% chance it came from Process C.

## 2) Entropy function

Entropy is trying to measure how “uncertain” a probability distribution is.

Quick intuition:
- If one outcome is basically guaranteed, entropy is low.
- If outcomes are more evenly spread, entropy is higher.

For a discrete distribution, the formula is:

**H = - Σ p · log2(p)**

(If p = 0, that term should contribute 0, since it is impossible.)

One assumption here is that the probabilities already form a valid distribution (they sum to 1).

In [71]:
def entropy(probs):
    """
    Compute Shannon entropy (base 2) for a discrete probability distribution.
    probs: array-like of probabilities that sum to 1
    """
    # Convert to numpy array so the math is consistent
    p = np.array(probs, dtype=float)

    # Remove zeros so we don't take log2(0)
    p = p[p > 0]

    # Shannon entropy in bits (log base 2)
    return -np.sum(p * np.log2(p))

### Quick sanity check

A uniform distribution should have higher entropy than a “peaked” distribution.

So I’ll test:
- uniform: [0.5, 0.5]
- peaked: [0.9, 0.1]

In [74]:
# These are simple tests just to confirm the function behaves how I expect
entropy([0.5, 0.5]), entropy([0.9, 0.1])

(1.0, 0.4689955935892812)

## 3) Calculate entropies for X and Y

X is uniform across 5 outcomes, so I expect its entropy to be relatively high.

Y is more uneven (one outcome has 0.4 and another has 0.3), so I expect its entropy to be lower than X.

In [77]:
# Probabilities given in the assignment table
pX = [0.2, 0.2, 0.2, 0.2, 0.2]
pY = [0.1, 0.4, 0.1, 0.3, 0.1]

# Use the entropy function for both distributions
HX = entropy(pX)
HY = entropy(pY)

HX, HY

(2.321928094887362, 2.046439344671015)

In [79]:
# Print the entropy values in a clean format (bits)
print(f"H(X) = {HX:.4f}")
print(f"H(Y) = {HY:.4f}")

H(X) = 2.3219
H(Y) = 2.0464


### Interpretation (compare H(X) vs H(Y))

The results are:

- **H(X) = 2.3219**
- **H(Y) = 2.0464**

Since X is perfectly uniform, every outcome is equally likely. That means there is more uncertainty about which value will occur, which increases the entropy.

Distribution **Y** places more probability on a few outcomes (0.4 and 0.3). Because the outcomes are less evenly spread, the distribution is more predictable, which lowers the entropy.

Bayes’ Theorem let me reverse the conditionals from the tree, which is useful anytime I want “cause given evidence.”  
Entropy then gave me a numeric way to compare uncertainty, and it matched the intuition that uniform distributions are the most uncertain.